# Análise dos Resultados

Lê `resultados/medicoes.csv` e `resultados/medicoes_poda.csv`, computa resumos estatísticos e gera todos os gráficos comparativos.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 220, "font.size": 12})

MEDICOES_PATH      = Path("resultados/medicoes.csv")
MEDICOES_PODA_PATH = Path("resultados/medicoes_poda.csv")

## 1. Carregamento dos dados

In [ ]:
def carregar(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["estourou_timeout"] = df["estourou_timeout"].astype(str).str.lower() == "true"
    return df

df      = carregar(MEDICOES_PATH)
df_poda = carregar(MEDICOES_PODA_PATH)

print(f"medicoes.csv      : {len(df)} linhas")
print(f"medicoes_poda.csv : {len(df_poda)} linhas")
df.head()

## 2. Resumo estatístico

Agrega as repetições em **média + desvio-padrão** por `(algoritmo, tipo_instancia, n)`.

In [ ]:
def resumir(df: pd.DataFrame) -> pd.DataFrame:
    validas = df[~df["estourou_timeout"]]
    grp = validas.groupby(["algoritmo", "tipo_instancia", "n"])
    resumo = grp.agg(
        execucoes_validas=("repeticao", "count"),
        tempo_medio_s=("tempo_s", "mean"),
        tempo_desvio_s=("tempo_s", "std"),
        memoria_media_mb=("pico_memoria_bytes", lambda x: x.mean() / 1e6),
        memoria_desvio_mb=("pico_memoria_bytes", lambda x: x.std() / 1e6),
        estados_medios=("estados", "mean"),
    ).reset_index()
    timeouts = (
        df[df["estourou_timeout"]]
        .groupby(["algoritmo", "tipo_instancia", "n"])
        .size()
        .reset_index(name="timeouts")
    )
    resumo = resumo.merge(timeouts, on=["algoritmo", "tipo_instancia", "n"], how="left")
    resumo["timeouts"] = resumo["timeouts"].fillna(0).astype(int)
    return resumo

resumo      = resumir(df)
resumo_poda = resumir(df_poda)

print("=== Resumo – Backtracking vs. Meet in the Middle ===")
resumo

In [ ]:
print("=== Resumo – Efeito da Poda ===")
resumo_poda

## 3. Configurações visuais

In [ ]:
CORES = {
    "backtracking":           "#1f77b4",
    "meet_in_middle":         "#d62728",
    "backtracking_com_poda":  "#2ca02c",
    "backtracking_sem_poda":  "#7f7f7f",
}
ROTULOS = {
    "backtracking":           "Backtracking (com poda)",
    "meet_in_middle":         "Divisão e Conquista (MITM)",
    "backtracking_com_poda":  "Backtracking com poda",
    "backtracking_sem_poda":  "Backtracking sem poda",
}
ROTULOS_TIPO = {
    "com_solucao":  "com solução",
    "sem_solucao":  "sem solução",
    "alvo_pequeno": "alvo pequeno",
    "alvo_grande":  "alvo grande",
}
MARCADORES = {
    "backtracking":           "o",
    "meet_in_middle":         "s",
    "backtracking_com_poda":  "^",
    "backtracking_sem_poda":  "v",
}

## 4. Tempo médio vs n por tipo de instância

In [ ]:
tipos = sorted(resumo["tipo_instancia"].unique())
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

for ax, tipo in zip(axes, tipos):
    sub = resumo[resumo["tipo_instancia"] == tipo]
    for alg, grupo in sub.groupby("algoritmo"):
        grupo = grupo.sort_values("n")
        ax.errorbar(
            grupo["n"], grupo["tempo_medio_s"],
            yerr=grupo["tempo_desvio_s"].fillna(0),
            fmt=f"{MARCADORES.get(alg, 'o')}-",
            color=CORES.get(alg),
            label=ROTULOS.get(alg, alg),
            capsize=3,
        )
    ax.set_yscale("log")
    ax.set_title(f"Instâncias {ROTULOS_TIPO.get(tipo, tipo)}")
    ax.set_xlabel("n (número de transações)")
    ax.set_ylabel("tempo médio (s)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("Tempo médio vs n – Backtracking vs. Meet in the Middle", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("graficos/tempo_por_classe.png", bbox_inches="tight")
plt.show()

## 5. Pico de memória vs n por tipo de instância

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()

for ax, tipo in zip(axes, tipos):
    sub = resumo[resumo["tipo_instancia"] == tipo]
    for alg, grupo in sub.groupby("algoritmo"):
        grupo = grupo.sort_values("n")
        ax.errorbar(
            grupo["n"], grupo["memoria_media_mb"],
            yerr=grupo["memoria_desvio_mb"].fillna(0),
            fmt=f"{MARCADORES.get(alg, 's')}-",
            color=CORES.get(alg),
            label=ROTULOS.get(alg, alg),
            capsize=3,
        )
    ax.set_yscale("log")
    ax.set_title(f"Instâncias {ROTULOS_TIPO.get(tipo, tipo)}")
    ax.set_xlabel("n (número de transações)")
    ax.set_ylabel("pico de memória (MB)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle("Pico de memória vs n – Backtracking vs. Meet in the Middle", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("graficos/memoria_por_classe.png", bbox_inches="tight")
plt.show()

## 6. Efeito da poda no Backtracking (estados explorados)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Estados explorados
sub = resumo_poda[resumo_poda["tipo_instancia"] == "com_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.plot(
        grupo["n"], grupo["estados_medios"],
        f"{MARCADORES.get(alg, '^')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("nós visitados (média)")
ax.set_title("Estados explorados – com poda vs. sem poda")
ax.grid(True, which="both", alpha=0.3)
ax.legend()


fig.tight_layout()
fig.savefig("graficos/efeito_poda.png", bbox_inches="tight")
plt.show()

## 7. Sensibilidade do Backtracking à entrada

In [ ]:
ESTILOS_TIPO = {
    "com_solucao":  ("o-",  "com solução"),
    "sem_solucao":  ("x--", "sem solução"),
    "alvo_pequeno": ("v:",  "alvo pequeno"),
    "alvo_grande":  ("d-.", "alvo grande"),
}

fig, ax = plt.subplots(figsize=(9, 5))

# Tempo
sub_bt = resumo[resumo["algoritmo"] == "backtracking"]
for tipo, (estilo, rotulo) in ESTILOS_TIPO.items():
    grupo = sub_bt[sub_bt["tipo_instancia"] == tipo].sort_values("n")
    if grupo.empty:
        continue
    ax.plot(grupo["n"], grupo["tempo_medio_s"], estilo, label=rotulo)
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("tempo médio (s)")
ax.set_title("Backtracking – tempo por tipo de instância")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig("graficos/sensibilidade_backtracking.png", bbox_inches="tight")
plt.show()

## 8. Comparativo direto: tempo e tempo por algoritmo

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = resumo[resumo["tipo_instancia"] == "com_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.errorbar(
        grupo["n"], grupo["memoria_media_mb"],
        yerr=grupo["memoria_desvio_mb"].fillna(0),
        fmt=f"{MARCADORES.get(alg, 'o')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
        capsize=4,
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("pico de memória (MB)")
ax.set_title("Backtracking vs. Meet in the Middle – instâncias com solução")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = resumo[resumo["tipo_instancia"] == "sem_solucao"]
for alg, grupo in sub.groupby("algoritmo"):
    grupo = grupo.sort_values("n")
    ax.errorbar(
        grupo["n"], grupo["tempo_medio_s"],
        yerr=grupo["tempo_desvio_s"].fillna(0),
        fmt=f"{MARCADORES.get(alg, 'o')}-",
        color=CORES.get(alg),
        label=ROTULOS.get(alg, alg),
        capsize=4,
    )
ax.set_yscale("log")
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("tempo médio (s) – escala log")
ax.set_title("Backtracking vs. Meet in the Middle – instâncias sem solução")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Tabelas de resumo detalhadas

In [ ]:
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_rows", 80)

print("=== Resumo – Backtracking vs. Meet in the Middle ===")
display(
    resumo
    .sort_values(["tipo_instancia", "algoritmo", "n"])
    [["algoritmo", "tipo_instancia", "n",
      "execucoes_validas", "timeouts",
      "tempo_medio_s", "tempo_desvio_s",
      "memoria_media_mb", "estados_medios"]]
    .reset_index(drop=True)
)

In [ ]:
print("=== Resumo – Efeito da Poda ===")
display(
    resumo_poda
    .sort_values(["algoritmo", "n"])
    [["algoritmo", "tipo_instancia", "n",
      "execucoes_validas", "timeouts",
      "tempo_medio_s", "tempo_desvio_s",
      "estados_medios"]]
    .reset_index(drop=True)
)

## 10. Fator de redução da poda

In [ ]:
pivot_estados = resumo_poda.pivot_table(
    index="n",
    columns="algoritmo",
    values="estados_medios",
).dropna()

pivot_estados["fator_reducao"] = (
    pivot_estados["backtracking_sem_poda"] / pivot_estados["backtracking"]
)
print("Fator de redução de estados (sem poda / com poda):")
display(pivot_estados[["backtracking", "backtracking_sem_poda", "fator_reducao"]].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(pivot_estados.index.astype(str), pivot_estados["fator_reducao"], color="#2ca02c", alpha=0.8)
ax.set_xlabel("n (número de transações)")
ax.set_ylabel("fator de redução (vs)")
ax.set_title("Quantas vezes a poda reduz os estados explorados")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()